# Week 5: Self-Correcting RAG (The "Glass Box" Agent)
## From Data Engineer to AI Architect
**Author:** Sreeram Raghav Nudurupati  
**Concept:** Moving beyond "Black Box" chains to "Glass Box" observable agents.

This notebook implements a **Corrective RAG (CRAG)** workflow. Unlike a linear RAG chain that fails silently, this agent:
1.  **Retrieves** knowledge from ArangoDB.
2.  **Grades** the quality of that knowledge (The "Critic").
3.  **Transforms** the query if the data is poor (The "Pivot").
4.  **Generates** a final answer only when it has valid facts.

### Phase 1: Infrastructure & "The Fuel"
We start by connecting to the database and initializing the AI models. 
**Architect Note:** We strip the ArangoDB URL to prevent the common `[HTTP 400]` error caused by trailing slashes.

In [ ]:
import os
from dotenv import load_dotenv
from arango import ArangoClient
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_arangodb import ArangoVector

# 1. Load Environment Variables
load_dotenv()

ARANGO_URL = os.getenv("ARANGO_URL", "http://localhost:8529").strip().strip("/")
ARANGO_PWD = os.getenv("ARANGO_PASSWORD", "").strip()
OPENAI_KEY = os.getenv("OPENAI_API_KEY")

if not ARANGO_PWD or not OPENAI_KEY:
    raise ValueError("❌ MISSING CREDENTIALS: Check your .env file.")

# 2. Connect to ArangoDB
client = ArangoClient(hosts=ARANGO_URL)
db = client.db("glass_box", username="root", password=ARANGO_PWD)

# 3. Initialize Models
# 'Fuel': text-embedding-3-small (1536 dims)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# 'Brain': gpt-4o-mini (Fast & Low Cost)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"✅ Connected to ArangoDB at {ARANGO_URL}")
print("✅ LLM & Embeddings Initialized.")

### Phase 2: The "Glass Box" Knowledge Base
We use the **Direct Constructor** pattern to connect to our existing `kb_nodes` collection. 
**Architect Note:** We explicitly set `embedding_dimension=1536` and `text_field="text"` to avoid auto-inference errors.

In [ ]:
# Initialize the Vector Store (Direct Connection)
vectorstore = ArangoVector(
    database=db,
    collection_name="kb_nodes",
    embedding=embeddings,
    # ARCHITECTURAL DECISION: Explicit configuration > Implicit defaults
    embedding_dimension=1536,  # Required for text-embedding-3-small
    embedding_field="vector",  # The pre-calculated vectors
    text_field="text",         # The human-readable content
    search_type="hybrid"       # Enable Hybrid Search (Vector + Keyword)
)

print("✅ ArangoVector Store Connected (Hybrid Mode).")

### Phase 3: The Surgical Data Patch
**Problem:** Ingestion pipelines often create schema fields but fail to populate them, leading to empty retrievals.
**Solution:** We inject a "Golden Record" directly into the `proj_alpha` document to ensure the agent has valid data to read.

In [ ]:
# The "Golden Record" for Project Alpha
project_alpha_content = """
PROJECT ALPHA SECURITY PROTOCOL
Confidentiality Level: High

1. Access Control: Multi-Factor Authentication (MFA) using hardware tokens is mandatory.
2. Encryption: Data at rest must be encrypted using AES-256.
3. Network: Implementation of 'Zero Trust' architecture for internal API calls.
4. Monitoring: Weekly vulnerability scans are mandatory.
"""

print("--- EXECUTING SURGICAL PATCH ---")

try:
    # Update the specific document by its key
    if db.collection('kb_nodes').has('proj_alpha'):
        db.collection('kb_nodes').update({
            '_key': 'proj_alpha', 
            'text': project_alpha_content.strip()
        })
        print(f"✅ SUCCESS: 'proj_alpha' document has been hydrated with text.")
    else:
        print("⚠️ WARNING: 'proj_alpha' node not found. Check Data Generator.")

except Exception as e:
    print(f"❌ Patch Failed: {e}")

### Phase 4: Defining the Agent Nodes
We define the three "Workers" of our factory:
1.  **Retriever:** Fetches data.
2.  **Grader:** Evaluates relevance.
3.  **Transformer:** Rewrites bad queries.

In [ ]:
from typing import List, TypedDict
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- STATE DEFINITION ---
class GraphState(TypedDict):
    question: str
    documents: List[str]
    is_relevant: str
    loop_count: int

# --- NODE 1: RETRIEVER ---
def retrieve_docs(state: GraphState):
    print("---NODE: RETRIEVING DOCUMENTS---")
    question = state["question"]
    # We retrieve the top 3 documents
    documents = vectorstore.similarity_search(question, k=3)
    # Extract just the content text to keep the state clean
    return {"documents": [doc.page_content for doc in documents]}

# --- NODE 2: GRADER (The Critic) ---
grader_system = """You are a strict grader assessing relevance of a retrieved document to a user question. 
If the document contains keywords or semantic meaning related to the question, grade it as 'yes'. 
Otherwise, grade it as 'no'. Give a binary score 'yes' or 'no' score to indicate relevance."""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", grader_system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)
retrieval_grader = grade_prompt | llm | StrOutputParser()

def grade_documents(state: GraphState):
    print("---NODE: GRADING DOCUMENTS---")
    question = state["question"]
    documents = state["documents"]
    
    # We only check the best document for efficiency
    score = retrieval_grader.invoke({"question": question, "document": documents[0]})
    return {"is_relevant": score}

# --- NODE 3: TRANSFORMER (The Pivot) ---
rewriter_system = """You are a query optimizer. Look at the input and reason about the underlying intent. 
Rephrase the question to be more specific or to use broader industry terms to find the answer in a database."""

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", rewriter_system),
        ("human", "Here is the initial question: \n\n {question} \n Formulate an improved version."),
    ]
)
question_rewriter = rewrite_prompt | llm | StrOutputParser()

def transform_query(state: GraphState):
    print("---NODE: TRANSFORMING QUERY---")
    question = state["question"]
    documents = state["documents"]
    loop_count = state.get("loop_count", 0)

    better_question = question_rewriter.invoke({"question": question})
    return {"question": better_question, "documents": documents, "loop_count": loop_count + 1}

### Phase 5: The State Machine (LangGraph)
We wire the nodes together. The conditional logic acts as the "Switch" on the track:
* If **Relevant** -> End.
* If **Not Relevant** -> Transform -> Retrieve (Loop).

In [ ]:
from langgraph.graph import END, StateGraph

# 1. Initialize Graph
workflow = StateGraph(GraphState)

# 2. Add Nodes
workflow.add_node("retrieve", retrieve_docs)
workflow.add_node("grade", grade_documents)
workflow.add_node("transform", transform_query)

# 3. Add Edges
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade")

# 4. Conditional Logic
def decide_to_generate(state):
    if state["is_relevant"] == "yes":
        return "useful"
    else:
        if state["loop_count"] >= 2: # Prevent infinite loops
            return "max_retries"
        return "not_useful"

workflow.add_conditional_edges(
    "grade",
    decide_to_generate,
    {
        "useful": END,
        "not_useful": "transform",
        "max_retries": END
    }
)
workflow.add_edge("transform", "retrieve")

# 5. Compile
app = workflow.compile()
print("✅ Self-Correcting State Machine is LIVE.")

### Phase 6: The "Glass Box" Execution Trace
We use `app.stream()` to watch the agent think. 
* **Success:** The `GRADE` node outputs "YES".
* **Self-Correction:** If "NO", the `TRANSFORM` node rewrites the query and tries again.

In [ ]:
# The Input
inputs = {
    "question": "What is the security protocol for Project Alpha?",
    "documents": [],
    "is_relevant": "",
    "loop_count": 0
}

print("---AGENT STARTING WORK (EXTENDED TRANSPARENCY)---\n")

for output in app.stream(inputs):
    for key, value in output.items():
        print(f"📍 NODE: {key.upper()}")
        
        if "question" in value:
            print(f"   👉 Query: \"{value['question']}\"")
        
        if "is_relevant" in value:
            print(f"   - Critic Grade: {value['is_relevant'].upper()}")
            
        if "documents" in value and value["documents"]:
            print(f"   - Documents Found: {len(value['documents'])}")
            # Show the retrieval snippet to prove the Data Patch worked
            print(f"   - Match Snippet: {value['documents'][0][:60]}...")
            
        if "loop_count" in value:
            print(f"   - Iteration: {value['loop_count']}")
            
    print("-" * 30)

print("\n---AGENT WORK COMPLETE---")